In [1]:
from nltk import pos_tag


def pos_tag_tokens(tokens: list[str]) -> list[tuple[str, str]]:
    return pos_tag(tokens)


def extract_noun_phrases(tokens: list[str]) -> list[str]:
    tagged = pos_tag_tokens(tokens)
    phrases, current = [], []
    for word, tag in tagged:
        if tag.startswith("NN"):
            current.append(word)
        else:
            if current:
                phrases.append(" ".join(current))
                current = []
    if current:
        phrases.append(" ".join(current))
    return phrases

In [2]:
# test pos_tag_tokens
tokens = ["The", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]
print(pos_tag_tokens(tokens))

[('The', 'DT'), ('quick', 'JJ'), ('brown', 'NN'), ('fox', 'NN'), ('jumps', 'VBZ'), ('over', 'IN'), ('the', 'DT'), ('lazy', 'JJ'), ('dog', 'NN')]


In [7]:
# test exxtra_noun_phrases
extract_noun_phrases(tokens)

['brown fox', 'dog']

In [12]:
from nltk import ne_chunk
from nltk.tree import Tree


def named_entities_nltk(tokens: list[str]) -> list[tuple[str, str]]:
    tagged = pos_tag(tokens)
    tree = ne_chunk(tagged)
    entities = []
    for subtree in tree:
        if isinstance(subtree, Tree):
            entity_text = " ".join(tok for tok, _ in subtree.leaves())
            entities.append((entity_text, subtree.label()))
    return entities

In [16]:
from nltk import word_tokenize

exemples = [
    "I bought this Samsung phone from Amazon last week",
    "John from customer service was very rude to me",
    "I ordered it while living in Paris",
]

for phrase in exemples:
    tokens = word_tokenize(phrase)
    entities = named_entities_nltk(tokens)
    print(f"{phrase}\n  -> entités : {entities}\n")

I bought this Samsung phone from Amazon last week
  -> entités : [('Samsung', 'ORGANIZATION'), ('Amazon', 'GPE')]

John from customer service was very rude to me
  -> entités : [('John', 'PERSON')]

I ordered it while living in Paris
  -> entités : [('Paris', 'GPE')]



In [17]:
import spacy

_SPACY_MODELS = {}


def _load_spacy_model(lang: str = "en"):
    model_names = {
        "en": "en_core_web_sm",
        "es": "es_core_news_sm",
        "de": "de_core_news_sm",
        "fr": "fr_core_news_sm",
    }
    if lang not in _SPACY_MODELS:
        _SPACY_MODELS[lang] = spacy.load(model_names[lang])
    return _SPACY_MODELS[lang]


def dependency_parse(text: str, lang: str = "en") -> list[dict]:
    nlp = _load_spacy_model(lang)
    doc = nlp(text)
    return [
        {"token": tok.text, "dep": tok.dep_, "head": tok.head.text, "pos": tok.pos_}
        for tok in doc
    ]

In [20]:
phrase = "the product is great but delivery was slow"
print(f"Phrase analysée : {phrase!r}\n")

resultats = dependency_parse(phrase, lang="en")

for r in resultats:
    print(f"  {r['token']:12} dep={r['dep']:10} head={r['head']:10} pos={r['pos']}")

print("\nCe qu'il faut vérifier :")
print('  - "slow" doit avoir dep=acomp et head=was')
print('  - "delivery" doit avoir dep=nsubj et head=was')
print('  -> ça confirme que "slow" se rattache bien à "delivery" via "was",')

Phrase analysée : 'the product is great but delivery was slow'

  the          dep=det        head=product    pos=DET
  product      dep=nsubj      head=is         pos=NOUN
  is           dep=ROOT       head=is         pos=AUX
  great        dep=acomp      head=is         pos=ADJ
  but          dep=cc         head=is         pos=CCONJ
  delivery     dep=nsubj      head=was        pos=NOUN
  was          dep=conj       head=is         pos=AUX
  slow         dep=acomp      head=was        pos=ADJ

Ce qu'il faut vérifier :
  - "slow" doit avoir dep=acomp et head=was
  - "delivery" doit avoir dep=nsubj et head=was
  -> ça confirme que "slow" se rattache bien à "delivery" via "was",
